In [224]:
import re
import numpy as np
import pandas as pd

In [225]:
with open("..\Corpus\corpus_RNN_Médine.txt", "r", encoding="utf-8", errors="replace") as f:
    corpus = f.read()

### Featuring will be removed from the Markov Chain as this model will only generate the lyrics of one artist, so the structure of the songs can't be of a featuring.

In [226]:
print("Nb of songs :",len(re.findall(r"<END>", corpus)), "\nNb of featurings :",len(re.findall(r"<BEGINNING>True.*?<END>", corpus, flags=re.DOTALL)))

Nb of songs : 250 
Nb of featurings : 97


In [227]:
corpus_solo = re.sub(r"<BEGINNING>True.*?<END>\n\n", " ", corpus, flags=re.DOTALL)

## Removing the songs with only one part identified

In [228]:
all_parts = [(m.group(1), m.start(), m.end()) for m in re.finditer(r"<(\w+)>", corpus_solo)]

for i in range(len(all_parts)-3) :
    nn = len(all_parts)-3 -i
    if all_parts[nn-2][0] == "BEGINNING" :
        if all_parts[nn-1][0] == "END_SECTION" :
            if all_parts[nn][0] == "END" :
                part_1_corp = corpus_solo[:all_parts[nn-2][1]]
                part_2_corp = corpus_solo[all_parts[nn][2]:]
                corpus_solo = part_1_corp+part_2_corp

### I will need to regroup each song and then count the number of occurences from each of my parts

In [229]:
all_parts = [(m.group(1), m.start(), m.end()) for m in re.finditer(r"<(\w+)>", corpus_solo)]

count = 0
beg = 0
endi =0

for i in all_parts :
    if i[0] == "BEGINNING" :
        count = 0
        beg = i[1]
    elif i[0] == "END_SECTION" :
        continue
    else : 
        if i[0] == "END" :
            count+=1
            endi = i[2]
            if count >=10 :
                print(count, beg, endi)
        else : 
            count+=1

#corpus_solo = corpus_solo[:beg] + corpus_solo[endi:]

10 14163 18396
10 80552 85147
10 94034 98112
11 141273 145267
10 399925 404523


Now we will need to annotate the parts by the number of times they appeared.

This will also help in finding errors in the lyrics or structure in the corpus

In [230]:
parts = re.findall(r"<(\w+)>", corpus_solo) #Identify each part
regrouped = " ".join(parts) #re.findall gave back a list, we need a full text to perform next modif
regrouped = re.sub("END_SECTION ", "", regrouped) 
splitted = regrouped.split("END")

splitted = [i.strip() for i in splitted[:-1]]

In [231]:
def add_cumsum_tags(parts):
    counts = {}
    result = []
    for p in parts:
        counts[p] = counts.get(p, 0)
        if counts[p] != 0 :
            result.append(f"{p}_{counts[p]}")
        else : 
            result.append(f"{p}")
        counts[p] += 1
    return result

def transform_to_num_structure(splitted) :
    #This function will serve only if I decide to aggregate more songs from other rapper because we need to have a sufficient nb of sample
    #or we will have a huge nb of single interaction such as couplet_3 refrain_3 ...
    new_splitted = []
    
    for split in splitted :
        interm = add_cumsum_tags(split.split(" "))
        new_splitted.append(interm)

    count_all = {}
        
    for elem in new_splitted :
        for length in range(len(elem)-1) :
            name = elem[length]+" "+elem[length+1]
            interm = count_all.get(name,0)+1
            count_all[name] = interm
    
    return new_splitted, count_all

In [232]:
new_split, count_all = transform_to_num_structure(splitted)

Maybe I will collect data structure of multiple artists and then aggregate to get a better Markov Chain but if I do this, it will be in the future not now

In [233]:
count_all

{'BEGINNING REFRAIN': 16,
 'REFRAIN COUPLET': 15,
 'COUPLET REFRAIN_1': 16,
 'REFRAIN_1 COUPLET_1': 13,
 'COUPLET_1 OUTRO': 7,
 'BEGINNING COUPLET': 72,
 'COUPLET REFRAIN': 59,
 'REFRAIN COUPLET_1': 60,
 'COUPLET_1 REFRAIN_1': 60,
 'REFRAIN_1 OUTRO': 20,
 'BEGINNING INTRO': 45,
 'INTRO COUPLET': 39,
 'COUPLET OUTRO': 11,
 'COUPLET PONT': 18,
 'PONT REFRAIN': 8,
 'COUPLET_1 PONT_1': 7,
 'PONT_1 REFRAIN_1': 3,
 'REFRAIN_1 COUPLET_2': 23,
 'COUPLET_2 REFRAIN_2': 16,
 'PONT COUPLET_1': 11,
 'INTRO REFRAIN': 5,
 'BEGINNING OUTRO': 2,
 'COUPLET COUPLET_1': 13,
 'REFRAIN OUTRO': 1,
 'REFRAIN PONT': 4,
 'COUPLET_1 REFRAIN_2': 12,
 'REFRAIN_2 COUPLET_3': 3,
 'COUPLET_3 REFRAIN_3': 2,
 'REFRAIN_2 OUTRO': 12,
 'REFRAIN_1 PONT_1': 3,
 'PONT_1 REFRAIN_2': 2,
 'PONT COUPLET': 2,
 'PONT_1 COUPLET_1': 1,
 'REFRAIN_2 COUPLET_2': 2,
 'REFRAIN REFRAIN_1': 4,
 'REFRAIN_1 COUPLET': 1,
 'COUPLET REFRAIN_2': 1,
 'COUPLET_1 COUPLET_2': 9,
 'COUPLET_2 PONT': 1,
 'PONT OUTRO': 3,
 'COUPLET_2 OUTRO': 8,
 'PONT_1

In [234]:
def count_each_interact(splitted) :
    
    for i in range(len(splitted)) :
        splitted[i] = splitted[i]+" END"

    good_parts = (" ".join(splitted)).split(" ")

    dict_glob = {}
    for i in range(len(good_parts)-1) :
        try : 
            dict_glob[f"{good_parts[i]} {good_parts[i+1]}"] += 1
        except :
            dict_glob[f"{good_parts[i]} {good_parts[i+1]}"] = 1

    beginn = {}
    intro = {}
    couplet = {}
    pont = {}
    refrain = {}
    outro = {}

    order = ["BEGINNING","INTRO", "COUPLET", "PONT", "REFRAIN", "OUTRO", "END"]
    list_dicti = [beginn, intro, couplet, pont, refrain, outro]
    for part in list_dicti :
        for j in order :
            part[j] = 0    
    
    for i in dict_glob.keys() :
        actual = i.split(" ")[0]
        next = i.split(" ")[1]

        if actual == "BEGINNING" :  
            beginn[next] = dict_glob[i]

        elif actual == "INTRO" :
            intro[next] = dict_glob[i]

        elif actual == "COUPLET" :
            couplet[next] = dict_glob[i]
        elif actual == "PONT" :
            pont[next] = dict_glob[i]
        elif actual == "REFRAIN" :
            refrain[next] = dict_glob[i]
        elif actual == "OUTRO" :
            outro[next] = dict_glob[i]

    return list_dicti

all_count = count_each_interact(splitted)

In [235]:
all_count = count_each_interact(splitted)

## Some errors can be found in the differents parts, we will remove what we think is impossible

In [236]:
all_count[0] #begin

{'BEGINNING': 0,
 'INTRO': 45,
 'COUPLET': 72,
 'PONT': 0,
 'REFRAIN': 16,
 'OUTRO': 2,
 'END': 0}

In [237]:
all_count[1] #intro

{'BEGINNING': 0,
 'INTRO': 0,
 'COUPLET': 40,
 'PONT': 1,
 'REFRAIN': 5,
 'OUTRO': 0,
 'END': 0}

In [238]:
all_count[2] #couplet

{'BEGINNING': 0,
 'INTRO': 0,
 'COUPLET': 24,
 'PONT': 27,
 'REFRAIN': 166,
 'OUTRO': 26,
 'END': 27}

In [239]:
all_count[3] #pont

{'BEGINNING': 0,
 'INTRO': 0,
 'COUPLET': 17,
 'PONT': 0,
 'REFRAIN': 16,
 'OUTRO': 4,
 'END': 0}

In [240]:
all_count[4] #refrain

{'BEGINNING': 0,
 'INTRO': 1,
 'COUPLET': 117,
 'PONT': 9,
 'REFRAIN': 6,
 'OUTRO': 33,
 'END': 43}

Intro isn't an error because I found songs where there was two intro and one for each rapper

In [241]:
all_count[5] #outro

{'BEGINNING': 0,
 'INTRO': 0,
 'COUPLET': 0,
 'PONT': 0,
 'REFRAIN': 0,
 'OUTRO': 0,
 'END': 65}

## Now we need to obtain the proba for each state to go to another

In [242]:
matrix = []
for i in all_count :
    inter_matrix = []
    for j in i :
        inter_matrix.append(i[j]/sum(i.values()))
    matrix.append(inter_matrix)

In [243]:
order = ["<BEGINNING>","<INTRO>", "<COUPLET>", "<PONT>", "<REFRAIN>", "<OUTRO>", "<END>"]

In [244]:
print(order[:])
print(matrix)

['<BEGINNING>', '<INTRO>', '<COUPLET>', '<PONT>', '<REFRAIN>', '<OUTRO>', '<END>']
[[0.0, 0.3333333333333333, 0.5333333333333333, 0.0, 0.11851851851851852, 0.014814814814814815, 0.0], [0.0, 0.0, 0.8695652173913043, 0.021739130434782608, 0.10869565217391304, 0.0, 0.0], [0.0, 0.0, 0.08888888888888889, 0.1, 0.6148148148148148, 0.0962962962962963, 0.1], [0.0, 0.0, 0.4594594594594595, 0.0, 0.43243243243243246, 0.10810810810810811, 0.0], [0.0, 0.004784688995215311, 0.5598086124401914, 0.0430622009569378, 0.028708133971291867, 0.15789473684210525, 0.20574162679425836], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0]]


In [245]:
def generate_a_song_structure(matrix) :
    
    song_struct = [0]

    index = [i for i, p in enumerate(matrix[0]) if p>0]
    proba = [p for p in matrix[0] if p > 0]

    cumsum = np.cumsum(proba)
    r = np.random.rand()

    idx = np.searchsorted(cumsum, r)
    selected_value = index[idx] 
    song_struct.append(selected_value)

    end = False

    while end == False :

        index = [i for i, p in enumerate(matrix[selected_value]) if p>0]
        proba = [p for p in matrix[selected_value] if p > 0]

        cumsum = np.cumsum(proba)
        r = np.random.rand()

        idx = np.searchsorted(cumsum, r)
        selected_value = index[idx]
        song_struct.append(selected_value)

        if selected_value == 6 :
            end = True

    print([order[i] for i in song_struct])

In [246]:
for run in range(5) :
    generate_a_song_structure(matrix)

['<BEGINNING>', '<REFRAIN>', '<COUPLET>', '<END>']
['<BEGINNING>', '<INTRO>', '<COUPLET>', '<PONT>', '<COUPLET>', '<REFRAIN>', '<COUPLET>', '<REFRAIN>', '<END>']
['<BEGINNING>', '<COUPLET>', '<REFRAIN>', '<REFRAIN>', '<COUPLET>', '<REFRAIN>', '<COUPLET>', '<REFRAIN>', '<COUPLET>', '<COUPLET>', '<REFRAIN>', '<PONT>', '<COUPLET>', '<PONT>', '<COUPLET>', '<END>']
['<BEGINNING>', '<COUPLET>', '<REFRAIN>', '<COUPLET>', '<OUTRO>', '<END>']
['<BEGINNING>', '<COUPLET>', '<REFRAIN>', '<COUPLET>', '<REFRAIN>', '<COUPLET>', '<OUTRO>', '<END>']


This seems ok for now. 

In [247]:
pd.concat((pd.DataFrame(matrix),pd.DataFrame(order).transpose()))

,0,1,2,3,4,5,6
0,0.0,0.333333,0.533333,0.0,0.118519,0.014815,0.0
1,0.0,0.0,0.869565,0.021739,0.108696,0.0,0.0
2,0.0,0.0,0.088889,0.1,0.614815,0.096296,0.1
3,0.0,0.0,0.459459,0.0,0.432432,0.108108,0.0
4,0.0,0.004785,0.559809,0.043062,0.028708,0.157895,0.205742
5,0.0,0.0,0.0,0.0,0.0,0.0,1.0
0,<BEGINNING>,<INTRO>,<COUPLET>,<PONT>,<REFRAIN>,<OUTRO>,<END>


So the matrix will be exported and used during the model generation of text

In [115]:
pd.concat((pd.DataFrame(matrix),pd.DataFrame(order).transpose())).to_csv("transition_matrix.csv", index=False)

In [222]:
for n in re.finditer(r"<\w+>\n<END_SECTION>", corpus_solo, flags=re.DOTALL) :
    print(n)

<re.Match object; span=(25246, 25266), match='<PONT>\n<END_SECTION>'>
